# RazorStrike v2 - clean-base multi-domain QLoRA (Colab Pro+ A100)

Base: `Qwen/Qwen3.6-35B-A3B` (clean, no merge, no abliteration).
Uncensoring is done via the `uncensor` data family, not weight ablation.

Flow: install -> auth -> clone -> build+push dataset -> train QLoRA -> (high-RAM) merge+push.
Runtime: **A100 (40GB)** for training. The merge step needs a **high-RAM** runtime (~70GB), not the GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
# Expect an A100 with ~40GB. A T4 (16GB) CANNOT load the 35B in 4-bit.

In [ ]:
!pip -q install -U "transformers>=4.57" "peft>=0.14" "bitsandbytes>=0.44" \
    "accelerate>=1.0" "datasets>=3.0" huggingface_hub

In [ ]:
import os
from huggingface_hub import login
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    os.environ.setdefault('HF_TOKEN', '')  # or paste here
login(os.environ['HF_TOKEN'])

In [ ]:
%cd /content
![ -d razorstrike ] || git clone https://github.com/lancejames221b/razorstrike.git
%cd /content/razorstrike
!git pull --ff-only

## 1. Build + push the combined dataset (all 8 families)
Uncapped: real downloads (decompile-bench->40k, NuminaMath 40k, cyber ~30k, mythos ~8k, plus synthetic families). Pushes to `DATA_REPO`.

In [ ]:
import os
os.environ['DATA_REPO'] = 'lancejames221b/razorstrike-v2-sft'
os.environ['PUSH'] = '1'
!python -m scripts.build_dataset

## 2. Train the text QLoRA
4-bit base fits the 40GB A100. Watch the `[guard] trainable params` line - it must be a non-trivial %.

In [ ]:
import os
os.environ['BASE_REPO']    = 'Qwen/Qwen3.6-35B-A3B'
os.environ['DATA_REPO']    = 'lancejames221b/razorstrike-v2-sft'
os.environ['ADAPTER_REPO'] = 'lancejames221b/razorstrike-v2-offsec-lora'
os.environ['OUT_DIR']      = '/content/adapter'
os.environ['MAXLEN']       = '4096'
os.environ['TARGET_MLP']   = '0'   # 1 = also adapt the 256 MoE experts (large)
# os.environ['RESUME'] = '1'       # resume from checkpoint after a disconnect
!python -m scripts.train_lora

## 3. Merge + publish (HIGH-RAM runtime, not the GPU)
35B bf16 merge needs ~70GB RAM. Switch to a high-RAM runtime; the adapter is safe on HF from step 2. Skippable if you serve base+adapter.

In [ ]:
import os
os.environ['BASE_REPO']    = 'Qwen/Qwen3.6-35B-A3B'
os.environ['ADAPTER_DIR']  = '/content/adapter'
os.environ['MERGED_REPO']  = 'lancejames221b/razorstrike-v2'
!python -m scripts.merge_push